In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json
import re
import shutil

we remove the bad files

In [2]:
DATA_DIR = Path("C:\\Users\\anton\\Documents\\AUEB_Thesis\\Empirical_Study\\spy_trades_new\\spy_trades_new")   
BAD_DIR = DATA_DIR / "_warning_files"
BAD_DIR.mkdir(exist_ok=True)

def extract_date_from_filename(path: Path):
    m = re.search(r"(\d{8})", path.stem)
    if not m:
        return None
    return pd.to_datetime(m.group(1), format="%Y%m%d", errors="coerce")

warning_start = pd.Timestamp("1993-01-04")
warning_end = pd.Timestamp("1993-01-28")

files = sorted(DATA_DIR.glob("*.h5"))

to_move = []
skipped = []

for f in files:
    dt = extract_date_from_filename(f)
    if dt is None or pd.isna(dt):
        skipped.append(f.name)
        continue

    if warning_start <= dt <= warning_end:
        to_move.append((f, dt))

print("files to move:", len(to_move))
print("files skipped due to bad filename/date:", len(skipped))
print("first few to move:", [x[0].name for x in to_move[:10]])

moved = []

for f, dt in to_move:
    dst = BAD_DIR / f.name
    shutil.move(str(f), str(dst))
    moved.append({"file": f.name, "date": str(dt.date())})

print("moved files:", len(moved))

files to move: 0
files skipped due to bad filename/date: 0
first few to move: []
moved files: 0


In [ ]:
DATA_DIR = Path(r"C:\\Users\\anton\\Documents\\AUEB_Thesis\\Empirical_Study\\spy_trades_new\\spy_trades_new")

OUTPUT_DIR = Path("./preprocessed_paper15_clean_391")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# You decided to keep 391 bars, including trades exactly at 16:00.
# This assumes m_index runs 571, 572, ..., 961.
EXPECTED_MIN_INDEX = np.arange(571, 962)  # 391 bars

TRAIN_START = "2013-01-01"
TRAIN_END = "2018-12-31"

VAL_START = "2019-01-01"
VAL_END = "2020-12-31"

TEST_START = "2021-01-01"
TEST_END = "2022-12-31"


# Alternative runs:
DATASET_KEY = "SPY/in_tr_1m/Trades"
PRICE_COL = "close_price" #this is the price column used in this preprocessing. for compatibility reasons with previous runs the name in the rest of the code remains vwap

# DATASET_KEY = "SPY/in_tr_1m/IID_trades"
# PRICE_COL = "close_price"

# DATASET_KEY = "SPY/in_tr_1m/IID_trades"
# PRICE_COL = "vwap"

#MAX_DAILY_VWAP_RATIO = 2.0     # reject days where VWAP is >2x or <0.5x daily median
#MAX_ABS_MINUTE_RETURN = 0.05   # reject days with any abs minute log return > 5%

In [51]:
trend_feature_cols = [
    # very local return, kept only as context
    "ret_1",

    # smoothed return direction
    "ret_mean_15",
    "ret_mean_30",
    "ret_mean_60",
    "ret_mean_120",

    # cumulative price trend over past windows
    "momentum_15",
    "momentum_30",
    "momentum_60",
    "momentum_120",

    # trend normalized by volatility
    "trend_strength_15",
    "trend_strength_30",
    "trend_strength_60",
    "trend_strength_120",

    # volatility controls
    "ret_std_30",
    "ret_std_60",
    "ret_std_120",
    "rv_30",
    "rv_60",
    "rv_120",
]

In [5]:
files = sorted(DATA_DIR.glob("*.h5"))

print("Number of files:", len(files))
print(files[:5])

Number of files: 8233
[WindowsPath('C:/Users/anton/Documents/AUEB_Thesis/Empirical_Study/spy_trades_new/spy_trades_new/19930129.h5'), WindowsPath('C:/Users/anton/Documents/AUEB_Thesis/Empirical_Study/spy_trades_new/spy_trades_new/19930201.h5'), WindowsPath('C:/Users/anton/Documents/AUEB_Thesis/Empirical_Study/spy_trades_new/spy_trades_new/19930202.h5'), WindowsPath('C:/Users/anton/Documents/AUEB_Thesis/Empirical_Study/spy_trades_new/spy_trades_new/19930203.h5'), WindowsPath('C:/Users/anton/Documents/AUEB_Thesis/Empirical_Study/spy_trades_new/spy_trades_new/19930204.h5')]


In [ ]:
def read_h5_file(path: Path, dataset_key: str = DATASET_KEY, price_col: str = PRICE_COL) -> pd.DataFrame:
    """
    Reads the selected dataset from the new H5 structure.


    The rest of the notebook expects a column called 'vwap',
    so we rename the selected price column to 'vwap'.
    """

    df = pd.read_hdf(path, key=dataset_key).copy()

    required = {"m_index", price_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {missing} in {dataset_key}")

    keep_cols = ["m_index", price_col]

    if "tot_vlm" in df.columns:
        keep_cols.append("tot_vlm")

    df = df[keep_cols].copy()

    # Rename selected price column to vwap so the old pipeline works unchanged
    df = df.rename(columns={price_col: "vwap"})

    # Optional: keep volume if available
    if "tot_vlm" in df.columns:
        df["tot_vlm"] = pd.to_numeric(df["tot_vlm"], errors="coerce")

    df["m_index"] = pd.to_numeric(df["m_index"], errors="coerce")
    df["vwap"] = pd.to_numeric(df["vwap"], errors="coerce")

    return df

In [8]:
def validate_day_relaxed(df: pd.DataFrame):
    required_cols = {"m_index", "vwap"}
    missing = required_cols - set(df.columns)
    if missing:
        return False, f"missing columns: {sorted(missing)}"

    if df.empty:
        return False, "empty dataframe"

    if df["m_index"].isna().any():
        return False, "m_index has missing values"

    if df["vwap"].isna().any():
        return False, "vwap has missing values"

    try:
        m_index = pd.to_numeric(df["m_index"], errors="raise").astype(int).to_numpy()
    except Exception:
        return False, "m_index cannot be converted to int"

    vwap = pd.to_numeric(df["vwap"], errors="coerce")
    if vwap.isna().any():
        return False, "vwap contains non-numeric values"

    if (vwap <= 0).any():
        return False, "observed vwap contains non-positive values"

    if len(np.unique(m_index)) != len(df):
        return False, "duplicate m_index values"

    allowed = set(EXPECTED_MIN_INDEX)
    observed = set(m_index)

    extra = sorted(observed - allowed)
    if len(extra) > 0:
        return False, f"m_index outside expected range; extras={extra[:10]}"

    return True, "ok"

In [9]:
def validate_day_prices(raw: pd.DataFrame, max_ratio=2.0):
    df = raw.copy()

    if "vwap" not in df.columns:
        return False, "missing vwap"

    df["vwap"] = pd.to_numeric(df["vwap"], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["vwap"])
    df = df[df["vwap"] > 0]

    if df.empty:
        return False, "no valid positive vwap"

    med = df["vwap"].median()

    if not np.isfinite(med) or med <= 0:
        return False, "bad median vwap"

    ratio = df["vwap"] / med

    min_ratio = ratio.min()
    max_ratio_observed = ratio.max()

    if max_ratio_observed > max_ratio or min_ratio < 1 / max_ratio:
        return False, (
            f"extreme vwap relative to daily median: "
            f"min_ratio={min_ratio:.4f}, max_ratio={max_ratio_observed:.4f}, median={med:.4f}"
        )

    return True, "ok"

In [10]:
def build_full_day_with_fill(df: pd.DataFrame, prev_day_last_vwap: float | None = None):
    """
    Build a full 571..961 daily grid.

    Logic:
    - observed minutes stay as they are
    - internal gaps are filled with same-day forward fill
    - leading missing minutes are filled with previous day's last valid VWAP if available
    - if no previous day's last VWAP is available, fallback to same-day backfill
    """

    out = df.copy()

    if "m_index" not in out.columns or "vwap" not in out.columns:
        raise ValueError("missing required columns: m_index / vwap")

    out = out[["m_index", "vwap"]].copy()

    # numeric coercion
    out["m_index"] = pd.to_numeric(out["m_index"], errors="coerce")
    out["vwap"] = pd.to_numeric(out["vwap"], errors="coerce")

    # drop unusable rows
    out = out.dropna(subset=["m_index", "vwap"]).copy()
    out["m_index"] = out["m_index"].astype(int)

    # keep only expected minute range and positive observed prices
    out = out[(out["m_index"] >= 571) & (out["m_index"] <= 961)].copy()
    out = out[out["vwap"] > 0].copy()

    if len(out) == 0:
        raise ValueError("no valid observed rows remain after cleaning")

    # if duplicates exist, keep the last observed row for each minute
    out = out.sort_values("m_index").drop_duplicates(subset="m_index", keep="last")

    # mark observed rows
    out["trade_flag"] = 1

    # reindex to full day
    expected_idx = pd.Index(np.arange(571, 962), name="m_index")
    out = out.set_index("m_index").reindex(expected_idx)

    out["trade_flag"] = out["trade_flag"].fillna(0).astype(int)

    # raw provider-style convention
    out["vwap_zero_filled"] = out["vwap"].fillna(0.0)

    # start with within-day forward fill
    out["vwap_ffill"] = out["vwap"].ffill()

    # fill leading missing values with previous day's last VWAP if available
    if prev_day_last_vwap is not None and pd.notna(prev_day_last_vwap) and prev_day_last_vwap > 0:
        leading_mask = out["vwap_ffill"].isna()
        if leading_mask.any():
            out.loc[leading_mask, "vwap_ffill"] = prev_day_last_vwap

    # final fallback: if first day or no previous-day anchor, use same-day bfill
    out["vwap_ffill"] = out["vwap_ffill"].bfill()

    # final sanity check
    if out["vwap_ffill"].isna().any():
        n_bad = int(out["vwap_ffill"].isna().sum())
        raise ValueError(f"vwap_ffill still has NaNs: {n_bad}")

    return out.reset_index()

In [ ]:
def make_features(day_df: pd.DataFrame, prev_day_last_vwap: float | None = None):
    df = day_df.copy().sort_values("m_index").reset_index(drop=True)

    #  validate price
    df["vwap_ffill"] = pd.to_numeric(df["vwap_ffill"], errors="coerce")
    df["vwap_ffill"] = df["vwap_ffill"].replace([np.inf, -np.inf], np.nan)

    if (df["vwap_ffill"] <= 0).any():
        raise ValueError(
            f"vwap_ffill has nonpositive values: {(df['vwap_ffill'] <= 0).sum()}"
        )

    if df["vwap_ffill"].isna().any():
        raise ValueError(
            f"vwap_ffill still has NaNs: {df['vwap_ffill'].isna().sum()}"
        )

    #  log price
    df["log_price_ffill"] = np.log(df["vwap_ffill"])

    if df["log_price_ffill"].isna().any() or np.isinf(df["log_price_ffill"]).any():
        raise ValueError("log_price_ffill contains invalid values")

    #  one-minute log return, including overnight gap
    if prev_day_last_vwap is not None and prev_day_last_vwap > 0:
        prev_log = np.log(prev_day_last_vwap)

        temp_series = pd.concat(
            [pd.Series([prev_log]), df["log_price_ffill"]]
        ).reset_index(drop=True)

        df["ret_1"] = temp_series.diff().iloc[1:].reset_index(drop=True)
    else:
        df["ret_1"] = (
            df["log_price_ffill"]
            .diff()
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0.0)
        )

    df["sq_ret_1"] = df["ret_1"] ** 2
    df["abs_ret_1"] = df["ret_1"].abs()

    
    df["y"] = df["ret_1"]

    windows = [15, 30, 60, 120]

    for w in windows:
        
        df[f"ret_mean_{w}"] = df["ret_1"].rolling(w, min_periods=w).mean()

        # volatility
        df[f"ret_std_{w}"] = df["ret_1"].rolling(w, min_periods=w).std(ddof=0)
        df[f"rv_{w}"] = df["sq_ret_1"].rolling(w, min_periods=w).sum()

        # cumulative price trend
        df[f"momentum_{w}"] = df["log_price_ffill"] - df["log_price_ffill"].shift(w)

        # normalized trend strength
        df[f"trend_strength_{w}"] = (
            df[f"momentum_{w}"] / (df[f"ret_std_{w}"] * np.sqrt(w) + 1e-8)
        )

    df[trend_feature_cols] = df[trend_feature_cols].replace([np.inf, -np.inf], np.nan)
    df[trend_feature_cols] = df[trend_feature_cols].fillna(0.0)

    return df

In [ ]:
prev_day_last_vwap = None
records = []
logs = []

for path in files:
    dt = extract_date_from_filename(path)

    try:
        raw = read_h5_file(path, dataset_key=DATASET_KEY, price_col=PRICE_COL)

        ok, reason = validate_day_relaxed(raw)
        if not ok:
            logs.append({
                "file": path.name,
                "date": str(dt.date()) if dt is not None and not pd.isna(dt) else "",
                "dataset_key": DATASET_KEY,
                "price_col": PRICE_COL,
                "status": "invalid",
                "reason": reason
            })
            continue

        if dt is None or pd.isna(dt):
            logs.append({
                "file": path.name,
                "date": "",
                "dataset_key": DATASET_KEY,
                "price_col": PRICE_COL,
                "status": "invalid",
                "reason": "could not parse date from filename"
            })
            continue

        full_day = build_full_day_with_fill(
            raw,
            prev_day_last_vwap=prev_day_last_vwap
        )

        
        feat = make_features(full_day, prev_day_last_vwap=prev_day_last_vwap)
        

        prev_day_last_vwap = float(full_day["vwap_ffill"].iloc[-1])

        feat.insert(0, "date", dt.normalize())
        feat.insert(1, "source_file", path.name)
        feat.insert(2, "dataset_key", DATASET_KEY)
        feat.insert(3, "price_col", PRICE_COL)

        records.append(feat)

        logs.append({
            "file": path.name,
            "date": str(dt.date()),
            "dataset_key": DATASET_KEY,
            "price_col": PRICE_COL,
            "status": "ok",
            "reason": "",
            "observed_minutes": int(full_day["trade_flag"].sum()),
            "missing_minutes": int((1 - full_day["trade_flag"]).sum()),
        })

    except Exception as e:
        logs.append({
            "file": path.name,
            "date": str(dt.date()) if dt is not None and not pd.isna(dt) else "",
            "dataset_key": DATASET_KEY,
            "price_col": PRICE_COL,
            "status": "error",
            "reason": repr(e)
        })

log_df = pd.DataFrame(logs).sort_values(["date", "file"]).reset_index(drop=True)

if len(records) == 0:
    raise ValueError("No valid records were created.")

panel_df = (
    pd.concat(records, ignore_index=True)
    .sort_values(["date", "m_index"])
    .reset_index(drop=True)
)

print("panel_df:", panel_df.shape)
print(log_df["status"].value_counts(dropna=False))

panel_df: (3219103, 34)
status
ok    8233
Name: count, dtype: int64


In [54]:
rows_per_day = panel_df.groupby("date").size()

print("Rows per day:")
print(rows_per_day.describe())
print(rows_per_day.value_counts().head())

bad_rows_per_day = rows_per_day[rows_per_day != len(EXPECTED_MIN_INDEX)]
print("bad rows-per-day days:", len(bad_rows_per_day))
display(bad_rows_per_day.head())

Rows per day:
count    8233.0
mean      391.0
std         0.0
min       391.0
25%       391.0
50%       391.0
75%       391.0
max       391.0
dtype: float64
391    8233
Name: count, dtype: int64
bad rows-per-day days: 0


Series([], dtype: int64)

In [55]:
TRAIN_START = "2000-01-01"
TRAIN_END = "2014-12-31"

VAL_START = "2015-01-01"
VAL_END = "2019-12-31"

TEST_START = "2020-01-01"
TEST_END = "2025-12-31"


train_df = panel_df[
    #(panel_df["date"] >= TRAIN_START) &
    (panel_df["date"] <= TRAIN_END)
].copy()

val_df = panel_df[
    (panel_df["date"] >= VAL_START) &
    (panel_df["date"] <= VAL_END)
].copy()

test_df = panel_df[
    (panel_df["date"] >= TEST_START) &
    (panel_df["date"] <= TEST_END)
].copy()

print("train days:", train_df["date"].nunique(), "rows:", len(train_df))
print("val days:", val_df["date"].nunique(), "rows:", len(val_df))
print("test days:", test_df["date"].nunique(), "rows:", len(test_df))

print("train:", train_df["date"].min(), train_df["date"].max())
print("val:", val_df["date"].min(), val_df["date"].max())
print("test:", test_df["date"].min(), test_df["date"].max())

train days: 5522 rows: 2159102
val days: 1258 rows: 491878
test days: 1453 rows: 568123
train: 1993-01-29 00:00:00 2014-12-31 00:00:00
val: 2015-01-02 00:00:00 2019-12-31 00:00:00
test: 2020-01-02 00:00:00 2025-10-13 00:00:00


In [56]:
feature_cols = trend_feature_cols

missing_features = [c for c in feature_cols if c not in panel_df.columns]

print("num features:", len(feature_cols))
print("missing:", missing_features)
print(feature_cols)

num features: 19
missing: []
['ret_1', 'ret_mean_15', 'ret_mean_30', 'ret_mean_60', 'ret_mean_120', 'momentum_15', 'momentum_30', 'momentum_60', 'momentum_120', 'trend_strength_15', 'trend_strength_30', 'trend_strength_60', 'trend_strength_120', 'ret_std_30', 'ret_std_60', 'ret_std_120', 'rv_30', 'rv_60', 'rv_120']


In [ ]:
scale_cols = feature_cols

# Standardize the data
train_means = train_df[scale_cols].mean()
train_stds = train_df[scale_cols].std().replace(0, 1.0)

train_scaled = train_df.copy()
val_scaled = val_df.copy()
test_scaled = test_df.copy()

train_scaled[scale_cols] = (train_scaled[scale_cols] - train_means) / train_stds
val_scaled[scale_cols] = (val_scaled[scale_cols] - train_means) / train_stds
test_scaled[scale_cols] = (test_scaled[scale_cols] - train_means) / train_stds

# weighting from previous runs 
"""
# 1. THE FAST FEATURES (The Eject Button)
# These get the heavy 2.0 weight (4x voting power in MSE)
price_weight = 2.0
fast_directional_cols = [ 
    "mean_6", "left_mean_6", "right_mean_6"
]
price_weight_fast = 2.0

# 2. THE SLOW FEATURES (The Trend Anchor)
# These get the compromise 1.5 weight (2.25x voting power in MSE)
slow_directional_cols = [
    "mean_14", "left_mean_14", "right_mean_14"
]
price_weight_slow = 1.7

# Apply the weights
for df in [train_scaled, val_scaled, test_scaled]:
    df["y"] *= price_weight
    df[fast_directional_cols] *= price_weight_fast
    df[slow_directional_cols] *= price_weight_slow

print(f"Success: Fast features are {price_weight_fast**2}x important. Slow features are {price_weight_slow**2}x important. y is {price_weight**2}x important.")
# -----------------------------

# Verify the changes: Fast stds should be ~2.0, Slow stds should be ~1.5
display(train_scaled[scale_cols].describe().T[["mean", "std"]])
"""

'\n# 1. THE FAST FEATURES (The Eject Button)\n# These get the heavy 2.0 weight (4x voting power in MSE)\nprice_weight = 2.0\nfast_directional_cols = [ \n    "mean_6", "left_mean_6", "right_mean_6"\n]\nprice_weight_fast = 2.0\n\n# 2. THE SLOW FEATURES (The Trend Anchor)\n# These get the compromise 1.5 weight (2.25x voting power in MSE)\nslow_directional_cols = [\n    "mean_14", "left_mean_14", "right_mean_14"\n]\nprice_weight_slow = 1.7\n\n# Apply the weights\nfor df in [train_scaled, val_scaled, test_scaled]:\n    df["y"] *= price_weight\n    df[fast_directional_cols] *= price_weight_fast\n    df[slow_directional_cols] *= price_weight_slow\n\nprint(f"Success: Fast features are {price_weight_fast**2}x important. Slow features are {price_weight_slow**2}x important. y is {price_weight**2}x important.")\n# -----------------------------\n\n# Verify the changes: Fast stds should be ~2.0, Slow stds should be ~1.5\ndisplay(train_scaled[scale_cols].describe().T[["mean", "std"]])\n'

In [58]:
def to_sequence_array(df: pd.DataFrame, feature_cols, seq_len=len(EXPECTED_MIN_INDEX)):
    X = []
    dates = []
    observed_minutes = []
    missing_minutes = []

    for dt, g in df.groupby("date", sort=True):
        g = g.sort_values("m_index")

        if len(g) != seq_len:
            continue

        X.append(g[feature_cols].to_numpy(dtype=np.float32))
        dates.append(dt)

        if "observed_minutes" in g.columns:
            observed_minutes.append(int(g["observed_minutes"].iloc[0]))
        else:
            observed_minutes.append(np.nan)

        if "missing_minutes" in g.columns:
            missing_minutes.append(int(g["missing_minutes"].iloc[0]))
        else:
            missing_minutes.append(np.nan)

    X = np.stack(X).astype(np.float32)
    dates = np.array(dates)
    observed_minutes = np.array(observed_minutes)
    missing_minutes = np.array(missing_minutes)

    return X, dates, observed_minutes, missing_minutes

In [59]:
X_train, dates_train, obs_train, miss_train = to_sequence_array(train_scaled, feature_cols)
X_val, dates_val, obs_val, miss_val = to_sequence_array(val_scaled, feature_cols)
X_test, dates_test, obs_test, miss_test = to_sequence_array(test_scaled, feature_cols)

X_train = np.clip(np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0), -10, 10)
X_val = np.clip(np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0), -10, 10)
X_test = np.clip(np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0), -10, 10)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

assert X_train.shape[1] == len(EXPECTED_MIN_INDEX)
assert X_val.shape[1] == len(EXPECTED_MIN_INDEX)
assert X_test.shape[1] == len(EXPECTED_MIN_INDEX)

assert X_train.shape[2] == len(feature_cols)
assert X_val.shape[2] == len(feature_cols)
assert X_test.shape[2] == len(feature_cols)

print("X_train NaN:", np.isnan(X_train).any())
print("X_train inf:", np.isinf(X_train).any())

print("X_val NaN:", np.isnan(X_val).any())
print("X_val inf:", np.isinf(X_val).any())

print("X_test NaN:", np.isnan(X_test).any())
print("X_test inf:", np.isinf(X_test).any())

X_train: (5522, 391, 19)
X_val: (1258, 391, 19)
X_test: (1453, 391, 19)
X_train NaN: False
X_train inf: False
X_val NaN: False
X_val inf: False
X_test NaN: False
X_test inf: False


In [20]:
for i, col in enumerate(feature_cols):
    print(
        f"{i:02d} {col}: "
        f"train min={X_train[:, :, i].min():.4f}, "
        f"train max={X_train[:, :, i].max():.4f}, "
        f"val min={X_val[:, :, i].min():.4f}, "
        f"val max={X_val[:, :, i].max():.4f}, "
        f"test min={X_test[:, :, i].min():.4f}, "
        f"test max={X_test[:, :, i].max():.4f}"
    )

00 y: train min=-10.0000, train max=10.0000, val min=-10.0000, val max=10.0000, test min=-10.0000, test max=10.0000
01 abs_change: train min=-0.6792, train max=10.0000, val min=-0.6792, val max=10.0000, test min=-0.6792, test max=10.0000
02 prev_abs_change: train min=-0.6777, train max=10.0000, val min=-0.6777, val max=10.0000, test min=-0.6777, test max=10.0000
03 mean_6: train min=-10.0000, train max=10.0000, val min=-10.0000, val max=10.0000, test min=-10.0000, test max=10.0000
04 std_6: train min=-1.0081, train max=10.0000, val min=-1.0081, val max=10.0000, test min=-1.0081, test max=10.0000
05 left_mean_6: train min=-10.0000, train max=10.0000, val min=-10.0000, val max=10.0000, test min=-10.0000, test max=10.0000
06 left_std_6: train min=-0.8197, train max=10.0000, val min=-0.8197, val max=10.0000, test min=-0.8197, test max=10.0000
07 right_mean_6: train min=-10.0000, train max=10.0000, val min=-10.0000, val max=10.0000, test min=-10.0000, test max=10.0000
08 right_std_6: train 

In [60]:
import numpy as np

def clipping_summary(X, name):
    n_total = X.size
    n_at_neg10 = np.sum(X == -10)
    n_at_pos10 = np.sum(X == 10)
    n_at_bounds = n_at_neg10 + n_at_pos10

    print(f"{name}")
    print(f"  total values      : {n_total}")
    print(f"  values == -10     : {n_at_neg10} ({100*n_at_neg10/n_total:.6f}%)")
    print(f"  values ==  10     : {n_at_pos10} ({100*n_at_pos10/n_total:.6f}%)")
    print(f"  values at bounds  : {n_at_bounds} ({100*n_at_bounds/n_total:.6f}%)")
    print()

clipping_summary(X_train, "X_train")
clipping_summary(X_val, "X_val")
clipping_summary(X_test, "X_test")

def clipping_by_feature(X, feature_cols, name):
    print(name)
    for i, col in enumerate(feature_cols):
        feat = X[:, :, i]
        n_total = feat.size
        n_neg10 = np.sum(feat == -10)
        n_pos10 = np.sum(feat == 10)
        n_bounds = n_neg10 + n_pos10
        print(
            f"{i:02d} {col}: "
            f"-10={n_neg10} ({100*n_neg10/n_total:.4f}%), "
            f"+10={n_pos10} ({100*n_pos10/n_total:.4f}%), "
            f"total={n_bounds} ({100*n_bounds/n_total:.4f}%)"
        )
    print()

clipping_by_feature(X_train, feature_cols, "X_train")
clipping_by_feature(X_val, feature_cols, "X_val")
clipping_by_feature(X_test, feature_cols, "X_test")

X_train
  total values      : 41022938
  values == -10     : 2621 (0.006389%)
  values ==  10     : 11613 (0.028309%)
  values at bounds  : 14234 (0.034698%)

X_val
  total values      : 9345682
  values == -10     : 102 (0.001091%)
  values ==  10     : 207 (0.002215%)
  values at bounds  : 309 (0.003306%)

X_test
  total values      : 10794337
  values == -10     : 651 (0.006031%)
  values ==  10     : 4064 (0.037649%)
  values at bounds  : 4715 (0.043680%)

X_train
00 ret_1: -10=626 (0.0290%), +10=695 (0.0322%), total=1321 (0.0612%)
01 ret_mean_15: -10=307 (0.0142%), +10=399 (0.0185%), total=706 (0.0327%)
02 ret_mean_30: -10=247 (0.0114%), +10=388 (0.0180%), total=635 (0.0294%)
03 ret_mean_60: -10=232 (0.0107%), +10=464 (0.0215%), total=696 (0.0322%)
04 ret_mean_120: -10=239 (0.0111%), +10=439 (0.0203%), total=678 (0.0314%)
05 momentum_15: -10=269 (0.0125%), +10=377 (0.0175%), total=646 (0.0299%)
06 momentum_30: -10=233 (0.0108%), +10=386 (0.0179%), total=619 (0.0287%)
07 momentum_6

In [ ]:
np.save(OUTPUT_DIR / "X_train_4.npy", X_train)
np.save(OUTPUT_DIR / "X_val_4.npy", X_val)
np.save(OUTPUT_DIR / "X_test_4.npy", X_test)

np.save(OUTPUT_DIR / "dates_train_4.npy", dates_train)
np.save(OUTPUT_DIR / "dates_val_4.npy", dates_val)
np.save(OUTPUT_DIR / "dates_test_4.npy", dates_test)

np.save(OUTPUT_DIR / "obs_train_4.npy", obs_train)
np.save(OUTPUT_DIR / "obs_val_4.npy", obs_val)
np.save(OUTPUT_DIR / "obs_test_4.npy", obs_test)

np.save(OUTPUT_DIR / "miss_train_4.npy", miss_train)
np.save(OUTPUT_DIR / "miss_val_4.npy", miss_val)
np.save(OUTPUT_DIR / "miss_test_4.npy", miss_test)

panel_df.to_parquet(OUTPUT_DIR / "panel_features_4.parquet", index=False)
log_df.to_csv(OUTPUT_DIR / "processing_log_4.csv", index=False)

scaler_info = {
    "feature_cols": feature_cols,
    "scale_cols": scale_cols,
    "means": train_means.to_dict(),
    "stds": train_stds.to_dict(),
    "seq_len": int(len(EXPECTED_MIN_INDEX)),
    "feature_dim": int(len(feature_cols)),
    "bars_per_day": int(len(EXPECTED_MIN_INDEX)),
    "m_index_start": int(EXPECTED_MIN_INDEX.min()),
    "m_index_end": int(EXPECTED_MIN_INDEX.max()),
    "base_series": "minute_log_return_from_vwap_ffill",
    "price_source": "vwap_ffill",
    "train_start": TRAIN_START,
    "train_end": TRAIN_END,
    "val_start": VAL_START,
    "val_end": VAL_END,
    "test_start": TEST_START,
    "test_end": TEST_END,
}

with open(OUTPUT_DIR / "scaler_info_4.json", "w") as f:
    json.dump(scaler_info, f, indent=2)

print("Saved to:", OUTPUT_DIR)

Saved to: preprocessed_paper15_clean_391
